# Residual signal analysis

This notebook is the main research component. Here, I explore whether residuals of stocks, that are factor (Fama-French 3 and 5) neutralised, still contain simple signals for future returns.

The structure of the notebook is as follows:
- Import the research panel and calculate rolling factor betas for each stock with 36 months moving window (24 months minimum).
- Define and compute simple signals based on the contemporaneous and lagged residuals.
- Observe the statistics of the signals (Spearman's IC and quintile sort).
- Winsorise the research panel and see if signals (if there are any) are still manifest.

In [1]:
import pandas as pd
import numpy as np

## Import research panel and compute rolling factor betas for each stock

In [2]:
panel_ff3 = pd.read_parquet("./parquets/monthly_top1500_and_ff3.parquet")
panel_ff5 = pd.read_parquet("./parquets/monthly_top1500_and_ff5.parquet")

In [3]:
panel_ff3.head(5)

,ticker,month,month_end_date,month_end_close,ret_1m,ret_fwd_1m,excess_ret_1m,mkt_rf,smb,hml,rf
0,A,2000-02,2000-02-29,67.4805,0.569267,0.001211,0.564967,0.0246,0.2125,-0.0977,0.0043
1,A,2000-03,2000-03-31,67.5622,0.001211,-0.147834,-0.003489,0.0521,-0.1741,0.0850,0.0047
2,A,2000-04,2000-04-28,57.5742,-0.147834,-0.169275,-0.152434,-0.0639,-0.0600,0.0645,0.0046
3,A,2000-05,2000-05-31,47.8283,-0.169275,0.001773,-0.174275,-0.0439,-0.0608,0.0459,0.0050
4,A,2000-06,2000-06-30,47.9131,0.001773,-0.447433,-0.002227,0.0468,0.1271,-0.0804,0.0040


In [4]:
panel_ff5.head(5)

,ticker,month,month_end_date,month_end_close,ret_1m,ret_fwd_1m,excess_ret_1m,mkt_rf,smb,hml,rmw,cma,rf
0,A,2000-02,2000-02-29,67.4805,0.569267,0.001211,0.564967,0.0245,0.1846,-0.0977,-0.1895,-0.0113,0.0043
1,A,2000-03,2000-03-31,67.5622,0.001211,-0.147834,-0.003489,0.0521,-0.1554,0.0850,0.1165,-0.0120,0.0047
2,A,2000-04,2000-04-28,57.5742,-0.147834,-0.169275,-0.152434,-0.0635,-0.0475,0.0645,0.0807,0.0563,0.0046
3,A,2000-05,2000-05-31,47.8283,-0.169275,0.001773,-0.174275,-0.0439,-0.0386,0.0459,0.0405,0.0147,0.0050
4,A,2000-06,2000-06-30,47.9131,0.001773,-0.447433,-0.002227,0.0468,0.0995,-0.0804,-0.0831,-0.0317,0.0040


In [5]:
# rolling Fama-French factors betas calculation
def estimate_rolling_ff_one_stock(df_stock, factors, window=36, min_obs=24):
    df_stock = df_stock.sort_values("month").reset_index(drop=True).copy()
    n = len(df_stock)

    # define name and regressors for ff3 and ff5
    name = f"ff{len(factors)}"
    y = df_stock["excess_ret_1m"].to_numpy(np.float64)
    F = [df_stock[f].to_numpy(np.float64) for f in factors]

    # only valid row if all ff data is present
    valid = np.isfinite(y)
    for x in F:
        valid &= np.isfinite(x)

    # design matrix
    Xfull = np.column_stack([np.ones(n), *F])

    # output variables
    alpha  = np.full(n, np.nan)
    betas  = np.full((n, len(factors)), np.nan)
    fitted = np.full(n, np.nan)
    resid  = np.full(n, np.nan)
    n_obs_used = np.full(n, np.nan)

    for t in range(n):
        # moving window
        start = max(0, t - window)
        mask = valid[start:t]

        if mask.sum() < min_obs:
            continue

        X = Xfull[start:t][mask]
        yy = y[start:t][mask]

        # solve OLS
        try:
            coef = np.linalg.solve(X.T @ X, X.T @ yy)
        except np.linalg.LinAlgError:
            coef, *_ = np.linalg.lstsq(X, yy, rcond=None)

        # assign regression coefficients
        alpha[t] = coef[0]
        betas[t] = coef[1:]
        n_obs_used[t] = mask.sum()

        # calculate the residual
        if valid[t]:
            fitted[t] = Xfull[t] @ coef
            resid[t] = y[t] - fitted[t]

    # output df
    out = df_stock.copy()
    out[f"alpha_{name}"] = alpha

    for i, f in enumerate(factors):
        out[f"beta_{f}"] = betas[:, i]

    # fitted excess returns and residuals
    out[f"{name}_fitted_excess_ret_1m"] = fitted
    out[f"{name}_resid_1m"] = resid
    out["n_obs_window"] = n_obs_used
    return out

# wrapper to create the residual panel by performing the above function for all stocks one-by-one
def estimate_rolling_ff_panel(df, factors, window=36, min_obs=24):
    results = []

    for _, df_stock in df.groupby("ticker", sort=False):
        results.append(
            estimate_rolling_ff_one_stock(
                df_stock,
                factors=factors,
                window=window,
                min_obs=min_obs,
            )
        )

    return pd.concat(results, axis=0, ignore_index=True)

In [6]:
# factor columns, forward return column for stats correlation later, and target excess return column
ff3_factors  = ["mkt_rf", "smb", "hml"]
ff5_factors  = ["mkt_rf", "smb", "hml", "rmw", "cma"]
reg_cols_ff3 = ["ticker", "month", *ff3_factors, "ret_fwd_1m", "excess_ret_1m"]
reg_cols_ff5 = ["ticker", "month", *ff5_factors, "ret_fwd_1m", "excess_ret_1m"]

beta_input_ff3 = panel_ff3[reg_cols_ff3].copy()
beta_input_ff5 = panel_ff5[reg_cols_ff5].copy()

# create the residual panels
ff3_resid_panel = estimate_rolling_ff_panel(
    beta_input_ff3,
    factors=ff3_factors,
    window=36,
    min_obs=24,
)

ff5_resid_panel = estimate_rolling_ff_panel(
    beta_input_ff5,
    factors=ff5_factors,
    window=36,
    min_obs=24,
)

In [7]:
ff3_resid_panel.loc[ff3_resid_panel.ff3_resid_1m.notna()].head()

,ticker,month,mkt_rf,smb,hml,ret_fwd_1m,excess_ret_1m,alpha_ff3,beta_mkt_rf,beta_smb,beta_hml,ff3_fitted_excess_ret_1m,ff3_resid_1m,n_obs_window
24,A,2002-02,-0.0230,-0.0108,0.0234,0.122315,0.025083,0.015194,2.197398,0.888775,-0.225298,-0.050217,0.075300,24.0
25,A,2002-03,0.0424,0.0409,0.0108,-0.140399,0.121015,0.019226,2.163763,0.856018,-0.274864,0.143012,-0.021997,25.0
26,A,2002-04,-0.0521,0.0582,0.0401,-0.122549,-0.141899,0.018861,2.137639,0.840152,-0.300386,-0.055659,-0.086240,26.0
27,A,2002-05,-0.0136,-0.0327,0.0152,-0.103199,-0.123949,0.018334,2.164918,0.761411,-0.368592,-0.041610,-0.082339,27.0
28,A,2002-06,-0.0723,0.0408,0.0037,-0.201463,-0.104499,0.012976,2.206185,0.833795,-0.272447,-0.113521,0.009022,28.0


In [8]:
# merge the relevant columns from residual panels to the reserach panel
new_cols_ff3 = [
    "ticker", "month",
    "alpha_ff3", "beta_mkt_rf", "beta_smb", "beta_hml",
    "ff3_fitted_excess_ret_1m", "ff3_resid_1m", "n_obs_window"
]

new_cols_ff5 = [
    "ticker", "month",
    "alpha_ff5", "beta_mkt_rf", "beta_smb", "beta_hml", "beta_rmw", "beta_cma",
    "ff5_fitted_excess_ret_1m", "ff5_resid_1m", "n_obs_window"
]

panel_ff3 = panel_ff3.merge(
    ff3_resid_panel[new_cols_ff3],
    on=["ticker", "month"],
    how="left",
    validate="one_to_one"
)

panel_ff5 = panel_ff5.merge(
    ff5_resid_panel[new_cols_ff5],
    on=["ticker", "month"],
    how="left",
    validate="one_to_one"
)

# show how many of the fitted betas are NaN
print(panel_ff3[["alpha_ff3", "beta_mkt_rf", "beta_smb", "beta_hml"]].isna().mean(), "\n")
print(panel_ff5[["alpha_ff5", "beta_mkt_rf", "beta_smb", "beta_hml", "beta_rmw", "beta_cma"]].isna().mean())

alpha_ff3      0.153094
beta_mkt_rf    0.153094
beta_smb       0.153094
beta_hml       0.153094
dtype: float64 

alpha_ff5      0.153166
beta_mkt_rf    0.153166
beta_smb       0.153166
beta_hml       0.153166
beta_rmw       0.153166
beta_cma       0.153166
dtype: float64


In [9]:
panel_ff3.loc[
    panel_ff3["ticker"] == "AAPL",
    [
        "ticker", "month", "excess_ret_1m",
        "mkt_rf", "smb", "hml",
        "alpha_ff3", "beta_mkt_rf", "beta_smb", "beta_hml",
        "ff3_fitted_excess_ret_1m", "ff3_resid_1m", "n_obs_window"
    ]
].tail(10)

,ticker,month,excess_ret_1m,mkt_rf,smb,hml,alpha_ff3,beta_mkt_rf,beta_smb,beta_hml,ff3_fitted_excess_ret_1m,ff3_resid_1m,n_obs_window
2043,AAPL,2025-06,0.018109,0.0486,0.0083,-0.0161,-0.002340,1.058776,0.109261,-0.405720,0.056556,-0.038447,36.0
2044,AAPL,2025-07,0.008298,0.0198,0.0027,-0.0127,-0.002208,1.010642,0.138814,-0.423445,0.023555,-0.015257,36.0
2045,AAPL,2025-08,0.115838,0.0184,0.0387,0.0442,-0.004056,0.935216,0.083477,-0.344289,0.001165,0.114672,36.0
2046,AAPL,2025-09,0.093580,0.0339,-0.0185,-0.0105,0.000055,0.911972,0.218050,-0.255952,0.029625,0.063955,36.0
2047,AAPL,2025-10,0.058116,0.0196,-0.0055,-0.0310,0.003612,0.862489,0.222799,-0.268983,0.027630,0.030487,36.0
2048,AAPL,2025-11,0.029365,-0.0013,0.0038,0.0376,0.004308,0.747458,0.315692,-0.446571,-0.012255,0.041620,36.0
2049,AAPL,2025-12,-0.028467,-0.0036,-0.0106,0.0242,0.005822,0.828551,0.200088,-0.340473,-0.007521,-0.020946,36.0
2050,AAPL,2026-01,-0.048538,0.0102,0.0221,0.0370,0.009800,0.675108,0.290437,-0.365132,0.009595,-0.058133,36.0
2051,AAPL,2026-02,NaN,NaN,NaN,NaN,0.006837,0.684745,0.192730,-0.366705,NaN,NaN,36.0
2052,AAPL,2026-03,NaN,NaN,NaN,NaN,0.005234,0.728660,0.146914,-0.344406,NaN,NaN,35.0


In [10]:
percentiles = [0.01, 0.05, 0.5, 0.95, 0.99]

# observe stats for the residual returns overall and monthly 
pd.DataFrame(
    {
        "ff3_resid_1m": panel_ff3["ff3_resid_1m"].describe(percentiles=percentiles),
        "ff5_resid_1m": panel_ff5["ff5_resid_1m"].describe(percentiles=percentiles),
        "ff3_resid_1m_monthly": panel_ff3.groupby("month")["ff3_resid_1m"].mean().describe(percentiles=percentiles),
        "ff5_resid_1m_monthly": panel_ff5.groupby("month")["ff5_resid_1m"].mean().describe(percentiles=percentiles),
    }
)

,ff3_resid_1m,ff5_resid_1m,ff3_resid_1m_monthly,ff5_resid_1m_monthly
count,374457.000000,374425.000000,743.000000,727.000000
mean,-0.002423,-0.002406,-0.002000,-0.002314
std,0.111567,0.115983,0.017112,0.016838
min,-3.326393,-3.062480,-0.080197,-0.080901
1%,-0.274202,-0.287457,-0.056799,-0.051259
5%,-0.152855,-0.159262,-0.028139,-0.026567
50%,-0.003829,-0.003475,-0.002309,-0.002658
95%,0.147932,0.154511,0.024477,0.021717
99%,0.294355,0.304018,0.048882,0.048683
max,16.124206,15.964944,0.092502,0.113283


## Define and compute signals based on residual returns

In [11]:
panel_ff3 = panel_ff3.sort_values(["ticker", "month"]).reset_index(drop=True)
panel_ff5 = panel_ff5.sort_values(["ticker", "month"]).reset_index(drop=True)

# create columns for lagged residuals
def add_resid_lags(df, resid_col, nlags=6):
    for lag in range(1, nlags + 1):
        df[f"{resid_col}_l{lag}"] = df.groupby("ticker")[resid_col].shift(lag)
    return df

# helper to streamline adding signals
def add_resid_signals(df, resid_col, signal_prefix="sig_resid", mom_lags=6, mom_rev_lags=3):
    df[f"{signal_prefix}_rev_1m"] = -df[resid_col]

    # 6 months lagged momentum signal
    lag_cols_1 = [f"{resid_col}_l{k}" for k in range(1, mom_lags + 1)]
    df[f"{signal_prefix}_mom_{mom_lags}m"] = df[lag_cols_1].sum(axis=1, min_count=mom_lags)

    # 3 months average lagged reversal signal
    lag_cols_2 = [f"{resid_col}_l{k}" for k in range(1, mom_rev_lags + 1)]
    df[f"{signal_prefix}_mom_rev_{mom_rev_lags}m"] = df[lag_cols_2].sum(axis=1, min_count=mom_rev_lags)

    return df

panel_ff3 = add_resid_lags(panel_ff3, "ff3_resid_1m")
panel_ff5 = add_resid_lags(panel_ff5, "ff5_resid_1m")

panel_ff3 = add_resid_signals(panel_ff3, "ff3_resid_1m")
panel_ff5 = add_resid_signals(panel_ff5, "ff5_resid_1m")

## Compute cross-sectional correlations between the signals and forward returns

In [12]:
# cross-sectional Spearman correlation between signal at month t and forward return at t+1
def monthly_spearman_ic(df, signal_col, target_col="ret_fwd_1m", min_n=50):
    tmp = df[["month", signal_col, target_col]].dropna().copy()
    grouped = tmp.groupby("month")

    # apply Spearman's IC with minimum 50 months
    out = grouped.apply(
        lambda x: pd.Series({
            "n": len(x),
            "ic": x[signal_col].corr(x[target_col], method="spearman")
                 if len(x) >= min_n else np.nan
        })
    ).reset_index()

    return out

# apply for all signals
ic_rev_ff3  = monthly_spearman_ic(panel_ff3, "sig_resid_rev_1m")
ic_rev_ff5  = monthly_spearman_ic(panel_ff5, "sig_resid_rev_1m")
ic_mom6_ff3 = monthly_spearman_ic(panel_ff3, "sig_resid_mom_6m")
ic_mom6_ff5 = monthly_spearman_ic(panel_ff5, "sig_resid_mom_6m")
ic_mom3_rev_ff3 = monthly_spearman_ic(panel_ff3, "sig_resid_mom_rev_3m")
ic_mom3_rev_ff5 = monthly_spearman_ic(panel_ff5, "sig_resid_mom_rev_3m")

ic_rev_ff3.loc[ic_rev_ff3.ic.notna()].head()

,month,n,ic
222,1982-09,50.0,0.245618
223,1982-10,50.0,-0.352989
224,1982-11,50.0,-0.031456
225,1982-12,50.0,0.111933
226,1983-01,51.0,-0.086606


In [13]:
# show stats for all the signals
pd.DataFrame(
    {
        "ic_rev_ff3": ic_rev_ff3["ic"].dropna().describe(),
        "ic_rev_ff5": ic_rev_ff5["ic"].dropna().describe(),
        "ic_mom6_ff3": ic_mom6_ff3["ic"].dropna().describe(),
        "ic_mom6_ff5": ic_mom6_ff5["ic"].dropna().describe(),
        "ic_mom3_rev_ff3": ic_mom3_rev_ff3["ic"].dropna().describe(),
        "ic_mom3_rev_ff5": ic_mom3_rev_ff5["ic"].dropna().describe()
    }
).T

,count,mean,std,min,25%,50%,75%,max
ic_rev_ff3,521.0,0.027152,0.115388,-0.352989,-0.044439,0.013694,0.100255,0.399161
ic_rev_ff5,521.0,0.025871,0.114396,-0.361537,-0.049110,0.016867,0.105161,0.351804
ic_mom6_ff3,516.0,0.001710,0.124410,-0.525402,-0.080370,-0.002627,0.080112,0.366297
ic_mom6_ff5,516.0,-0.000613,0.122620,-0.481471,-0.080229,-0.005513,0.075129,0.361131
ic_mom3_rev_ff3,519.0,-0.005391,0.115104,-0.420209,-0.083804,-0.004289,0.073036,0.413333
ic_mom3_rev_ff5,519.0,-0.004644,0.112154,-0.502639,-0.075840,-0.000219,0.068352,0.394851


In [14]:
# add a minimum monthly filter
ic_rev_ff3_500  = ic_rev_ff3.loc[ic_rev_ff3["n"] >= 500].copy()
ic_rev_ff5_500  = ic_rev_ff5.loc[ic_rev_ff5["n"] >= 500].copy()
ic_mom6_ff3_500  = ic_mom6_ff3.loc[ic_mom6_ff3["n"] >= 500].copy()
ic_mom6_ff5_500  = ic_mom6_ff5.loc[ic_mom6_ff5["n"] >= 500].copy()
ic_mom3_rev_ff3_500  = ic_mom3_rev_ff3.loc[ic_mom3_rev_ff3["n"] >= 500].copy()
ic_mom3_rev_ff5_500  = ic_mom3_rev_ff5.loc[ic_mom3_rev_ff5["n"] >= 500].copy()

pd.DataFrame(
    {
        "ic_rev_ff3_500":  ic_rev_ff3_500["ic"].dropna().describe(),
        "ic_rev_ff5_500":  ic_rev_ff5_500["ic"].dropna().describe(),
        "ic_mom6_ff3_500": ic_mom6_ff3_500["ic"].dropna().describe(),
        "ic_mom6_ff5_500": ic_mom6_ff5_500["ic"].dropna().describe(),
        "ic_mom3_rev_ff3_500": ic_mom3_rev_ff3_500["ic"].dropna().describe(),
        "ic_mom3_rev_ff5_500": ic_mom3_rev_ff5_500["ic"].dropna().describe()
    }
).T

,count,mean,std,min,25%,50%,75%,max
ic_rev_ff3_500,225.0,0.008948,0.098837,-0.328193,-0.049071,-0.000377,0.070167,0.310138
ic_rev_ff5_500,225.0,0.004222,0.098677,-0.290843,-0.062194,0.002131,0.065535,0.278394
ic_mom6_ff3_500,220.0,0.001363,0.103804,-0.318705,-0.067725,-0.003948,0.072836,0.292351
ic_mom6_ff5_500,220.0,0.004163,0.102669,-0.322844,-0.061215,0.001633,0.076582,0.290327
ic_mom3_rev_ff3_500,223.0,-0.001208,0.097117,-0.224554,-0.073123,-0.003377,0.071632,0.234274
ic_mom3_rev_ff5_500,223.0,0.002770,0.094512,-0.277795,-0.065133,0.007632,0.066063,0.288594


In [15]:
# make summary with t-statistics on IC dataframes
def ic_summary(ic_df):
    x = ic_df["ic"].dropna()
    tstat = x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))
    return pd.Series({
        "n_months": len(x),
        "mean_ic": x.mean(),
        "std_ic": x.std(ddof=1),
        "tstat": tstat,
    })

pd.DataFrame(
    {
        "ic_rev_ff3_500":  ic_summary(ic_rev_ff3_500),
        "ic_rev_ff5_500":  ic_summary(ic_rev_ff5_500),
        "ic_mom6_ff3_500": ic_summary(ic_mom6_ff3_500),
        "ic_mom6_ff5_500": ic_summary(ic_mom6_ff5_500),
        "ic_mom3_rev_ff3_500": ic_summary(ic_mom3_rev_ff3_500),
        "ic_mom3_rev_ff5_500": ic_summary(ic_mom3_rev_ff5_500)
    }
).T

,n_months,mean_ic,std_ic,tstat
ic_rev_ff3_500,225.0,0.008948,0.098837,1.358064
ic_rev_ff5_500,225.0,0.004222,0.098677,0.641775
ic_mom6_ff3_500,220.0,0.001363,0.103804,0.194716
ic_mom6_ff5_500,220.0,0.004163,0.102669,0.601463
ic_mom3_rev_ff3_500,223.0,-0.001208,0.097117,-0.185752
ic_mom3_rev_ff5_500,223.0,0.002770,0.094512,0.437715


In [16]:
# compare with raw reversal
panel_ff3["sig_raw_rev_1m"] = -panel_ff3["ret_1m"]
panel_ff5["sig_raw_rev_1m"] = -panel_ff5["ret_1m"]

def compare_with_raw_reversal(df, df_ic, rev_name):
    ic_raw_rev = monthly_spearman_ic(df, "sig_raw_rev_1m", min_n=50)
    ic_raw_rev = ic_raw_rev.loc[ic_raw_rev["n"] >= 500].copy()

    common_months = set(ic_raw_rev["month"]).intersection(set(df_ic["month"]))

    out = pd.DataFrame(
        {
            f"{rev_name}_raw":  ic_summary(ic_raw_rev.loc[ic_raw_rev["month"].isin(common_months)]),
            f"{rev_name}":  ic_summary(df_ic.loc[df_ic["month"].isin(common_months)]),
        }
    )

    return out

pd.concat(
    [
        compare_with_raw_reversal(panel_ff3, ic_rev_ff3_500, "ic_rev_ff3_500"),
        compare_with_raw_reversal(panel_ff5, ic_rev_ff5_500, "ic_rev_ff5_500")
    ],
    axis=1
).T

,n_months,mean_ic,std_ic,tstat
ic_rev_ff3_500_raw,225.0,0.003534,0.134335,0.394659
ic_rev_ff3_500,225.0,0.008948,0.098837,1.358064
ic_rev_ff5_500_raw,225.0,0.003534,0.134335,0.394659
ic_rev_ff5_500,225.0,0.004222,0.098677,0.641775


In [17]:
# alternative ranking by quintile sorts
def make_quintile_sorts(df, signal_col, ret_col="ret_fwd_1m", min_n=500):
    tmp = df[["month", "ticker", signal_col, ret_col]].dropna().copy()

    # helper for month-by-month quintile rank
    def _one_month(x):
        n = len(x)
        if n < min_n:
            return None

        x = x.copy()

        # rank first to avoid qcut issues from duplicate values
        x["_rank"] = x[signal_col].rank(method="first")
        x["quintile"] = pd.qcut(x["_rank"], 5, labels=[1, 2, 3, 4, 5])

        # get mean for each rank
        out = (
            x.groupby("quintile", observed=False)[ret_col]
            .mean()
            .rename("ew_ret")
            .reset_index()
        )
        out["month"] = x["month"].iloc[0]
        out["n"] = n
        return out

    pieces = []
    for month, x in tmp.groupby("month"):
        res = _one_month(x)
        if res is not None:
            pieces.append(res)

    qrets = pd.concat(pieces, ignore_index=True)
    return qrets

# summarise the quintile rank, t-statistics, and spread of Q5-Q1
# note: for reversal signals, Q5 is the most negative residual bucket
def summarise_qspread(qrets):
    wide = qrets.pivot(index="month", columns="quintile", values="ew_ret").copy()
    wide["Q5_Q1"] = wide[5] - wide[1]

    spread = wide["Q5_Q1"].dropna()
    tstat = spread.mean() / (spread.std(ddof=1) / np.sqrt(len(spread)))

    summary = pd.Series({
        "n_months": len(spread),
        "mean_Q5_Q1": spread.mean(),
        "std_Q5_Q1": spread.std(ddof=1),
        "tstat": tstat,
    })

    return summary

In [18]:
# compute and compare quintile ranks for one month raw and fitted reversals
q_raw_ff3 = make_quintile_sorts(panel_ff3, "sig_raw_rev_1m", min_n=500)
q_raw_ff5 = make_quintile_sorts(panel_ff5, "sig_raw_rev_1m", min_n=500)
q_resid_ff3 = make_quintile_sorts(panel_ff3, "sig_resid_rev_1m", min_n=500)
q_resid_ff5 = make_quintile_sorts(panel_ff5, "sig_resid_rev_1m", min_n=500)

pd.DataFrame(
    {
        "q_spread_raw_ff3": summarise_qspread(q_raw_ff3),
        "q_spread_raw_ff5": summarise_qspread(q_raw_ff5),
        "q_spread_resid_ff3": summarise_qspread(q_resid_ff3),
        "q_spread_resid_ff5": summarise_qspread(q_resid_ff5)
    }
).T

,n_months,mean_Q5_Q1,std_Q5_Q1,tstat
q_spread_raw_ff3,250.0,0.000597,0.040824,0.231358
q_spread_raw_ff5,250.0,0.000597,0.040824,0.231358
q_spread_resid_ff3,225.0,0.003572,0.031295,1.712261
q_spread_resid_ff5,225.0,0.002323,0.031794,1.096002


In [19]:
# mean quintile buckets of one month raw and fitted reversals
pd.DataFrame(
    {
        "q_mean_raw_ff3": q_raw_ff3.groupby("quintile")["ew_ret"].mean(),
        "q_mean_raw_ff5": q_raw_ff5.groupby("quintile")["ew_ret"].mean(),
        "q_mean_resid_ff3": q_resid_ff3.groupby("quintile")["ew_ret"].mean(),
        "q_mean_resid_ff5": q_resid_ff5.groupby("quintile")["ew_ret"].mean(),
    }
).T

quintile,1,2,3,4,5
q_mean_raw_ff3,0.009718,0.009052,0.009412,0.009730,0.010315
q_mean_raw_ff5,0.009718,0.009052,0.009412,0.009730,0.010315
q_mean_resid_ff3,0.007988,0.008307,0.009144,0.009700,0.011560
q_mean_resid_ff5,0.008952,0.008485,0.008712,0.009275,0.011275


## Winsorise forward returns in the research panel and observe statsistics persistence

In [20]:
# winsorise the forward returns
def winsorise_by_month(df, col, lower=0.05, upper=0.95):
    def _clip(x):
        lo = x.quantile(lower)
        hi = x.quantile(upper)
        return x.clip(lo, hi)
    return df.groupby("month")[col].transform(_clip)

panel_ff3["ret_fwd_1m_wins"] = winsorise_by_month(panel_ff3, "ret_fwd_1m")
panel_ff5["ret_fwd_1m_wins"] = winsorise_by_month(panel_ff5, "ret_fwd_1m")

# calculate Spearman's IC using winsorised forward returns as target
ic_rev_ff3_wins  = monthly_spearman_ic(panel_ff3, "sig_resid_rev_1m", target_col="ret_fwd_1m_wins")
ic_rev_ff5_wins  = monthly_spearman_ic(panel_ff5, "sig_resid_rev_1m", target_col="ret_fwd_1m_wins")

# filter again with 500 months minimum
ic_rev_ff3_wins_500 = ic_rev_ff3_wins.loc[ic_rev_ff3_wins["n"] >= 500]
ic_rev_ff5_wins_500 = ic_rev_ff5_wins.loc[ic_rev_ff5_wins["n"] >= 500]

pd.DataFrame(
    {
        "ic_rev_ff3_500":  ic_summary(ic_rev_ff3_500),
        "ic_rev_ff3_wins_500": ic_summary(ic_rev_ff3_wins_500),
        "ic_rev_ff5_500":  ic_summary(ic_rev_ff5_500),
        "ic_rev_ff5_wins_500": ic_summary(ic_rev_ff5_wins_500)
    }
).T

,n_months,mean_ic,std_ic,tstat
ic_rev_ff3_500,225.0,0.008948,0.098837,1.358064
ic_rev_ff3_wins_500,225.0,0.009007,0.098847,1.366846
ic_rev_ff5_500,225.0,0.004222,0.098677,0.641775
ic_rev_ff5_wins_500,225.0,0.004288,0.098655,0.652018


In [21]:
# redo the quintile sorts with the winsorised forward returns
q_rev_ff3_wins = make_quintile_sorts(
    panel_ff3, "sig_resid_rev_1m", ret_col="ret_fwd_1m_wins", min_n=500
)
q_rev_ff5_wins = make_quintile_sorts(
    panel_ff5, "sig_resid_rev_1m", ret_col="ret_fwd_1m_wins", min_n=500
)

pd.DataFrame(
    {
        "q_spread_resid_ff3": summarise_qspread(q_resid_ff3),
        "q_spread_resid_ff3_wins": summarise_qspread(q_rev_ff3_wins),
        "q_spread_resid_ff5": summarise_qspread(q_resid_ff5),
        "q_spread_resid_ff5_wins": summarise_qspread(q_rev_ff5_wins)
    }
).T

,n_months,mean_Q5_Q1,std_Q5_Q1,tstat
q_spread_resid_ff3,225.0,0.003572,0.031295,1.712261
q_spread_resid_ff3_wins,225.0,0.002714,0.025551,1.593025
q_spread_resid_ff5,225.0,0.002323,0.031794,1.096002
q_spread_resid_ff5_wins,225.0,0.001608,0.025391,0.950075
